# Colab setup: Search_AI_project (OTTO recsys)

Run this once at the start of every Colab session. It:
1. Mounts Drive and clones/syncs the repo there (so code + git history persist across sessions)
2. Installs the same Python environment as local (via `uv`)
3. Sets up Kaggle credentials and downloads the **full** OTTO dataset (~11GB) to Drive, so it's
   downloaded once and reused across every future session instead of re-downloading each time -
   this project will likely span many sessions (Two-Tower, DCN-V2, MMoE, ensembling), so paying
   the download cost once is worth Drive's slower-than-local I/O. Skips the download automatically
   if the file's already there.

Before running: make sure you have a GPU runtime (Runtime -> Change runtime type -> GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shaliqsv/Search_AI_project.git"
REPO_ROOT = Path('/content/drive/MyDrive/search_ai_project')

if not (REPO_ROOT / "pyproject.toml").exists():
    print(f"Repo not found at {REPO_ROOT} - cloning ...")
    !git clone "{REPO_URL}" "{REPO_ROOT}"
else:
    # see financebench_project's setup cell for why: self-heals a detached HEAD,
    # and --no-rebase --no-edit self-heals a diverged local/origin history
    # without hanging on an editor prompt inside a notebook `!` cell
    print(f"Repo already present at {REPO_ROOT} - syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

print(f"REPO_ROOT: {REPO_ROOT}")

## Install the environment

Colab already ships numpy/pandas/torch. We only need to add what it doesn't have. Installing
straight into Colab's own environment rather than a `uv` venv on the Drive mount deliberately -
a venv is thousands of small files, and Drive's FUSE mount is slow and occasionally flaky for
that pattern. The whole Colab runtime is already disposable/isolated, so skipping the extra
venv layer is the standard approach here, not a shortcut.

In [ ]:
!pip install -q polars duckdb lightgbm kaggle

import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
print("environment ready")

## Kaggle credentials

`google.colab.userdata` (the Secrets manager) is gated to Colab's own browser UI - it cannot work
over a remote-kernel connection (e.g. VS Code connected to a Colab runtime), which is what threw
`TimeoutException: Secrets can only be fetched when running from the Colab UI`.

Instead: the first time this runs, it asks for your username/key via a hidden prompt (never
written into this notebook's cells or output) and saves them to `kaggle.json` on Drive, **outside**
the cloned repo folder - so it can never accidentally get committed to git even by an overly broad
`git add`. Every run after the first loads silently from that file.

In [ ]:
import getpass
import json
import os
from pathlib import Path

KAGGLE_CREDS_PATH = Path('/content/drive/MyDrive/kaggle.json')  # outside REPO_ROOT, deliberately

if KAGGLE_CREDS_PATH.exists():
    creds = json.loads(KAGGLE_CREDS_PATH.read_text())
    print(f"Loaded saved credentials from {KAGGLE_CREDS_PATH}")
else:
    print("No saved credentials found - enter them once (kaggle.com/settings -> API -> ")
    print("Create New Token, open the downloaded kaggle.json to see both values)")
    creds = {
        "username": getpass.getpass("KAGGLE_USERNAME: "),
        "key": getpass.getpass("KAGGLE_KEY: "),
    }
    KAGGLE_CREDS_PATH.write_text(json.dumps(creds))
    KAGGLE_CREDS_PATH.chmod(0o600)
    print(f"Saved to {KAGGLE_CREDS_PATH} for future sessions")

os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"] = creds["key"]
print("Kaggle credentials set")

## Download the full OTTO dataset

Downloaded once to Drive (persists across sessions - see the note at the top). CC BY 4.0, sourced
from the dataset endpoint not the competition - see `_doc/decisions/D1-otto-license.md` for why
that distinction matters.

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/otto_data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

train_file = DATA_DIR / "otto-recsys-train.jsonl"
if train_file.exists():
    size_gb = train_file.stat().st_size / 1e9
    print(f"Already downloaded: {train_file} ({size_gb:.1f} GB) - skipping")
else:
    !kaggle datasets download -d otto/recsys-dataset -p "{DATA_DIR}" --unzip

!ls -lh "{DATA_DIR}"

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Next steps

With the full dataset at `/content/drive/MyDrive/otto_data/otto-recsys-train.jsonl` (persisted on
Drive), the Phase 1 notebook's sampling step can either be skipped (use the full file directly, now
that RAM/disk aren't the constraint they were locally) or re-run at a much higher keep-percentage.
From there, Phase 1-7 notebooks port over unchanged (same `src/ranking` code, same logic) - the only
difference is scale and that `ROOT` now resolves to this Drive-mounted repo instead of the local
Mac path.

Two-Tower training (PyTorch, GPU) is the next real piece of new work once the data's in place.